# 01 Single Factor MVP

## 目标

- 用一个轻量 notebook 跑通 `mom_20d` 单因子的第一轮研究闭环。
- 默认使用 mock `daily_panel`，先验证口径、流程和图表。
- 保留 `duckdb` 切换口，等真实 `daily_panel` 稳定后直接替换输入。

## 本轮边界

- 先不抽 `src/` 因子模块。
- 先不做行业中性化，只做市值中性化。
- 先不做正式回测、交易成本和实盘成交假设。


## 研究口径

- 因子：`mom_20d = close(T) / close(T-20) - 1`
- 信号时点：`T` 日收盘后生成信号
- 标签口径：close-to-close，不把 `T` 日收盘价当成可成交价

```text
forward_return_1d  = close(T+2)  / close(T+1) - 1
forward_return_5d  = close(T+6)  / close(T+1) - 1
forward_return_20d = close(T+21) / close(T+1) - 1
```

注意：这是一轮研究标签，不是严格可交易收益。后续如果接入 `open` 或 `vwap`，再升级成 next-open / next-vwap 口径。


In [ ]:
from pathlib import Path
import sys
import warnings

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

DATA_MODE = 'mock'  # 'mock' or 'duckdb'
DUCKDB_PATH = PROJECT_ROOT / 'data' / 'warehouse' / 'ashare.duckdb'
MAX_DUCKDB_DATES = 252
FACTOR_NAME = 'mom_20d'
RNG_SEED = 7

FACTOR_COLUMNS = [
    'factor_value_raw',
    'factor_value_zscore',
    'factor_value_size_neutral',
]
FORWARD_COLUMNS = [
    'forward_return_1d',
    'forward_return_5d',
    'forward_return_20d',
]
PRIMARY_FACTOR = 'factor_value_size_neutral'
PRIMARY_FORWARD = 'forward_return_5d'
PRIMARY_QUANTILE = 'quantile_size_neutral'

VERSION_LABELS = {
    'factor_value_raw': 'raw',
    'factor_value_zscore': 'zscore',
    'factor_value_size_neutral': 'size_neutral',
}

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_MODE: {DATA_MODE}')
print(f'DUCKDB_PATH exists: {DUCKDB_PATH.exists()}')


In [ ]:
def make_mock_daily_panel(n_dates=140, n_stocks=90, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    trade_dates = pd.bdate_range('2023-01-02', periods=n_dates)
    industries = np.array(['电子', '医药', '银行', '机械', '消费'])
    ts_codes = []
    for idx in range(n_stocks):
        if idx < n_stocks // 2:
            ts_codes.append(f'{600000 + idx:06d}.SH')
        else:
            ts_codes.append(f'{300000 + idx - n_stocks // 2:06d}.SZ')

    stock_industry = rng.choice(industries, size=n_stocks)
    size_anchor = rng.lognormal(mean=3.2, sigma=0.8, size=n_stocks)
    factor_skill = rng.normal(0.0, 1.0, size=n_stocks)
    market_wave = rng.normal(0.0, 0.008, size=n_dates)

    frames = []
    for idx, ts_code in enumerate(ts_codes):
        noise = rng.normal(0.0, 0.018, size=n_dates)
        seasonal = 0.002 * np.sin(np.linspace(0, 5 * np.pi, n_dates) + idx / 9)
        drift = 0.0008 * factor_skill[idx]
        daily_ret = drift + 0.35 * market_wave + seasonal + noise
        close = (12 + idx * 0.08) * np.exp(np.cumsum(daily_ret))
        market_cap = close * size_anchor[idx] * 1e8

        stock_frame = pd.DataFrame(
            {
                'trade_date': trade_dates.strftime('%Y%m%d'),
                'ts_code': ts_code,
                'close': close,
                'market_cap': market_cap,
                'is_suspend': False,
                'industry': stock_industry[idx],
            }
        )
        frames.append(stock_frame)

    panel = pd.concat(frames, ignore_index=True)

    suspend_idx = rng.choice(panel.index.to_numpy(), size=max(10, len(panel) // 120), replace=False)
    close_missing_idx = rng.choice(panel.index.to_numpy(), size=max(12, len(panel) // 150), replace=False)
    cap_missing_idx = rng.choice(panel.index.to_numpy(), size=max(12, len(panel) // 150), replace=False)
    bad_cap_idx = rng.choice(panel.index.to_numpy(), size=max(8, len(panel) // 220), replace=False)

    panel.loc[suspend_idx, 'is_suspend'] = True
    panel.loc[close_missing_idx, 'close'] = np.nan
    panel.loc[cap_missing_idx, 'market_cap'] = np.nan
    panel.loc[bad_cap_idx, 'market_cap'] = 0.0

    short_history_codes = ts_codes[-5:]
    panel.loc[
        panel['ts_code'].isin(short_history_codes) & (panel['trade_date'] < trade_dates[25].strftime('%Y%m%d')),
        ['close', 'market_cap'],
    ] = np.nan

    return panel.sort_values(['trade_date', 'ts_code']).reset_index(drop=True)


def load_duckdb_daily_panel(duckdb_path=DUCKDB_PATH, max_dates=MAX_DUCKDB_DATES):
    if not duckdb_path.exists():
        raise FileNotFoundError(f'DuckDB file not found: {duckdb_path}')

    sql = f"""
    WITH selected_dates AS (
        SELECT trade_date
        FROM (
            SELECT DISTINCT trade_date
            FROM daily_panel
            ORDER BY trade_date DESC
            LIMIT {int(max_dates)}
        )
    )
    SELECT
        p.trade_date,
        p.ts_code,
        CAST(p.close AS DOUBLE) AS close,
        CAST(p.total_mv AS DOUBLE) AS market_cap,
        COALESCE(CAST(p.is_suspended AS BOOLEAN), FALSE) AS is_suspend,
        COALESCE(p.sw_l1_name, p.stock_basic_industry, 'UNKNOWN') AS industry
    FROM daily_panel p
    INNER JOIN selected_dates d
        ON p.trade_date = d.trade_date
    ORDER BY p.trade_date, p.ts_code
    """

    with duckdb.connect(str(duckdb_path), read_only=True) as con:
        table_exists = con.execute(
            """
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_schema = 'main' AND table_name = 'daily_panel'
            """
        ).fetchone()[0]
        if table_exists == 0:
            raise RuntimeError('daily_panel table does not exist yet. Build it first or keep DATA_MODE=mock.')
        panel = con.execute(sql).fetchdf()

    panel['trade_date'] = panel['trade_date'].astype(str)
    panel['industry'] = panel['industry'].fillna('UNKNOWN')
    panel['is_suspend'] = panel['is_suspend'].fillna(False).astype(bool)
    return panel


def get_daily_panel(data_mode=DATA_MODE):
    if data_mode == 'mock':
        return make_mock_daily_panel()
    if data_mode == 'duckdb':
        return load_duckdb_daily_panel()
    raise ValueError(f'Unsupported DATA_MODE: {data_mode}')


def build_pit_warning_table():
    return pd.DataFrame(
        [
            ['close', 'check', '需要确认是否为研究可用价格口径，以及是否含未来复权影响'],
            ['market_cap', 'check', '需要确认 total_mv 是否可直接作为当日市值'],
            ['is_suspend', 'check', '需要确认是否完整覆盖当日停牌状态'],
            ['industry', 'risky', '未确认历史行业口径前，只用于展示，不用于行业中性化'],
            ['stock_status', 'risky', '不能只保留当前仍上市股票回看历史'],
            ['adj_factor', 'check', '真实数据接入时要重点确认是否存在未来信息'],
        ],
        columns=['field', 'status', 'note'],
    )


In [ ]:
def winsorize_mad(series, n=3.0):
    valid = series.dropna()
    if len(valid) < 5:
        return series
    median = valid.median()
    mad = (valid - median).abs().median()
    if pd.isna(mad) or mad == 0:
        return series
    scale = 1.4826 * mad
    lower = median - n * scale
    upper = median + n * scale
    return series.clip(lower=lower, upper=upper)


def zscore_series(series):
    valid = series.dropna()
    if len(valid) < 2:
        return pd.Series(np.nan, index=series.index)
    std = valid.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(0.0, index=series.index)
    return (series - valid.mean()) / std


def compute_mom_20d(panel):
    ordered = panel.sort_values(['ts_code', 'trade_date']).copy()
    ordered['mom_20d'] = ordered.groupby('ts_code')['close'].transform(lambda s: s / s.shift(20) - 1)
    return ordered


def compute_forward_returns(panel, horizons=(1, 5, 20)):
    ordered = panel.sort_values(['ts_code', 'trade_date']).copy()
    close_group = ordered.groupby('ts_code')['close']
    for horizon in horizons:
        ordered[f'forward_return_{horizon}d'] = close_group.shift(-(horizon + 1)) / close_group.shift(-1) - 1
    return ordered


def build_universe_mask(panel):
    panel = panel.copy()
    panel['has_close'] = panel['close'].notna()
    panel['has_market_cap'] = panel['market_cap'].fillna(0).gt(0)
    panel['has_factor_history'] = panel['mom_20d'].notna()
    panel['has_forward_label'] = panel['forward_return_20d'].notna()
    panel['universe_mask'] = (
        panel['has_close']
        & (~panel['is_suspend'].fillna(False))
        & panel['has_market_cap']
        & panel['has_factor_history']
        & panel['has_forward_label']
    )
    return panel


def neutralize_by_size(panel, factor_col='factor_value_zscore'):
    residuals = pd.Series(np.nan, index=panel.index, dtype='float64')
    for _, part in panel.groupby('trade_date'):
        mask = (
            part['universe_mask']
            & part[factor_col].notna()
            & part['market_cap'].notna()
            & part['market_cap'].gt(0)
        )
        if mask.sum() < 5:
            continue
        x = np.log(part.loc[mask, 'market_cap'].astype(float).to_numpy())
        y = part.loc[mask, factor_col].astype(float).to_numpy()
        design = np.column_stack([np.ones(len(x)), x])
        beta = np.linalg.lstsq(design, y, rcond=None)[0]
        resid = y - design @ beta
        resid_series = pd.Series(resid, index=part.index[mask])
        std = resid_series.std(ddof=0)
        if pd.notna(std) and std > 0:
            resid_series = (resid_series - resid_series.mean()) / std
        else:
            resid_series[:] = 0.0
        residuals.loc[resid_series.index] = resid_series
    return residuals


def assign_quantiles(panel, factor_col, quantiles=5):
    output = pd.Series(pd.NA, index=panel.index, dtype='Int64')
    for _, part in panel.groupby('trade_date'):
        mask = part['universe_mask'] & part[factor_col].notna()
        values = part.loc[mask, factor_col]
        if len(values) < quantiles:
            continue
        ranked = values.rank(method='first')
        bins = pd.qcut(ranked, q=quantiles, labels=list(range(1, quantiles + 1)))
        output.loc[values.index] = bins.astype('Int64')
    return output


def build_factor_data(panel):
    factor_data = compute_mom_20d(panel)
    factor_data = compute_forward_returns(factor_data)
    factor_data = build_universe_mask(factor_data)
    factor_data['factor_name'] = FACTOR_NAME
    factor_data['pit_warning'] = np.where(
        DATA_MODE == 'mock',
        'mock_only',
        'industry_display_only_until_historical_pit_is_validated',
    )

    factor_data['factor_value_raw'] = factor_data['mom_20d'].where(factor_data['universe_mask'])
    factor_data['factor_value_winsor'] = factor_data.groupby('trade_date')['factor_value_raw'].transform(winsorize_mad)
    factor_data['factor_value_zscore'] = factor_data.groupby('trade_date')['factor_value_winsor'].transform(zscore_series)
    factor_data['factor_value_size_neutral'] = neutralize_by_size(factor_data)

    factor_data['quantile_raw'] = assign_quantiles(factor_data, 'factor_value_raw')
    factor_data['quantile_zscore'] = assign_quantiles(factor_data, 'factor_value_zscore')
    factor_data['quantile_size_neutral'] = assign_quantiles(factor_data, 'factor_value_size_neutral')

    keep_columns = [
        'trade_date',
        'ts_code',
        'factor_name',
        'universe_mask',
        'factor_value_raw',
        'factor_value_zscore',
        'factor_value_size_neutral',
        'forward_return_1d',
        'forward_return_5d',
        'forward_return_20d',
        'quantile_raw',
        'quantile_zscore',
        'quantile_size_neutral',
        'market_cap',
        'industry',
        'pit_warning',
        'close',
        'is_suspend',
        'mom_20d',
        'has_close',
        'has_market_cap',
        'has_factor_history',
        'has_forward_label',
    ]
    return factor_data[keep_columns].sort_values(['trade_date', 'ts_code']).reset_index(drop=True)


In [ ]:
def spearman_corr(x, y):
    sample = pd.DataFrame({'x': x, 'y': y}).dropna()
    if len(sample) < 3:
        return np.nan
    x_rank = sample['x'].rank(method='average')
    y_rank = sample['y'].rank(method='average')
    return x_rank.corr(y_rank)


def compute_rank_ic(factor_data):
    rows = []
    base = factor_data[factor_data['universe_mask']].copy()
    if base.empty:
        return pd.DataFrame(
            columns=['trade_date', 'rank_ic', 'factor_version', 'forward_horizon']
        )
    for factor_col in FACTOR_COLUMNS:
        for forward_col in FORWARD_COLUMNS:
            daily_ic = (
                base.groupby('trade_date')
                .apply(lambda part: spearman_corr(part[factor_col], part[forward_col]))
                .rename('rank_ic')
                .reset_index()
            )
            daily_ic['factor_version'] = factor_col
            daily_ic['forward_horizon'] = forward_col
            rows.append(daily_ic)
    return pd.concat(rows, ignore_index=True)


def summarize_ic(ic_frame):
    clean = ic_frame.dropna(subset=['rank_ic']).copy()
    summary = (
        clean.groupby(['factor_version', 'forward_horizon'])['rank_ic']
        .agg(['count', 'mean', 'std'])
        .reset_index()
    )
    win_rate = (
        clean.groupby(['factor_version', 'forward_horizon'])['rank_ic']
        .apply(lambda s: (s > 0).mean())
        .reset_index(name='win_rate')
    )
    summary = summary.merge(win_rate, on=['factor_version', 'forward_horizon'], how='left')
    summary['ic_ir'] = summary['mean'] / summary['std'].replace(0, np.nan)
    return summary.sort_values(['forward_horizon', 'factor_version']).reset_index(drop=True)


def compute_quantile_returns(factor_data, factor_col=PRIMARY_FACTOR, forward_col=PRIMARY_FORWARD, quantile_col=PRIMARY_QUANTILE):
    sample = factor_data[
        factor_data['universe_mask']
        & factor_data[factor_col].notna()
        & factor_data[quantile_col].notna()
        & factor_data[forward_col].notna()
    ].copy()
    if sample.empty:
        empty_quantile = pd.DataFrame({'quantile': pd.Series(dtype='Int64'), 'avg_forward_return': pd.Series(dtype='float64')})
        empty_daily = pd.DataFrame({'trade_date': pd.Series(dtype='string'), 'quantile': pd.Series(dtype='Int64'), 'avg_forward_return': pd.Series(dtype='float64')})
        empty_long_short = pd.DataFrame({'trade_date': pd.Series(dtype='string'), 'long_short_return': pd.Series(dtype='float64'), 'cum_return': pd.Series(dtype='float64')})
        return empty_quantile, empty_daily, pd.DataFrame(), empty_long_short

    daily_quantile = (
        sample.groupby(['trade_date', quantile_col], as_index=False)[forward_col]
        .mean()
        .rename(columns={quantile_col: 'quantile', forward_col: 'avg_forward_return'})
    )

    quantile_summary = (
        daily_quantile.groupby('quantile', as_index=False)['avg_forward_return']
        .mean()
        .sort_values('quantile')
        .reset_index(drop=True)
    )

    quantile_pivot = daily_quantile.pivot(index='trade_date', columns='quantile', values='avg_forward_return').sort_index()
    for bucket in [1, 5]:
        if bucket not in quantile_pivot.columns:
            quantile_pivot[bucket] = np.nan
    long_short = pd.DataFrame(
        {
            'trade_date': quantile_pivot.index,
            'long_short_return': quantile_pivot[5] - quantile_pivot[1],
        }
    )
    long_short['cum_return'] = (1.0 + long_short['long_short_return'].fillna(0.0)).cumprod() - 1.0

    return quantile_summary, daily_quantile, quantile_pivot, long_short


def build_stability_table(ic_frame, factor_col=PRIMARY_FACTOR, forward_col=PRIMARY_FORWARD):
    sample = ic_frame[
        (ic_frame['factor_version'] == factor_col)
        & (ic_frame['forward_horizon'] == forward_col)
    ].dropna(subset=['rank_ic']).sort_values('trade_date')
    split = len(sample) // 2
    first_half = sample.iloc[:split]
    second_half = sample.iloc[split:]
    rows = []
    for label, part in [('first_half', first_half), ('second_half', second_half)]:
        if len(part) == 0:
            rows.append({'period': label, 'count': 0, 'mean_ic': np.nan, 'win_rate': np.nan, 'ic_ir': np.nan})
            continue
        std = part['rank_ic'].std(ddof=0)
        rows.append(
            {
                'period': label,
                'count': len(part),
                'mean_ic': part['rank_ic'].mean(),
                'win_rate': (part['rank_ic'] > 0).mean(),
                'ic_ir': part['rank_ic'].mean() / std if std else np.nan,
            }
        )
    return pd.DataFrame(rows)


def plot_factor_diagnostics(factor_data, ic_frame, quantile_summary, long_short):
    sample = factor_data[factor_data['universe_mask']].copy()
    primary_ic = ic_frame[
        (ic_frame['factor_version'] == PRIMARY_FACTOR)
        & (ic_frame['forward_horizon'] == PRIMARY_FORWARD)
    ].dropna(subset=['rank_ic']).sort_values('trade_date')

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    if sample.empty:
        for ax in axes.flat:
            ax.axis('off')
            ax.text(0.5, 0.5, 'Not enough valid rows for plotting', ha='center', va='center', fontsize=11)
        plt.tight_layout()
        plt.show()
        return

    axes[0, 0].hist(sample['factor_value_raw'].dropna(), bins=35, color='#4c78a8', alpha=0.85)
    axes[0, 0].set_title('Raw mom_20d distribution')

    axes[0, 1].hist(sample['factor_value_size_neutral'].dropna(), bins=35, color='#f58518', alpha=0.85)
    axes[0, 1].set_title('Size-neutral distribution')

    if primary_ic.empty:
        axes[0, 2].axis('off')
        axes[0, 2].text(0.5, 0.5, 'Not enough rows for Rank IC', ha='center', va='center')
    else:
        axes[0, 2].plot(pd.to_datetime(primary_ic['trade_date']), primary_ic['rank_ic'], color='#54a24b', linewidth=1.2)
        axes[0, 2].axhline(0.0, color='black', linewidth=0.8, linestyle='--')
        axes[0, 2].set_title('Rank IC time series (5d)')

    if primary_ic.empty:
        axes[1, 0].axis('off')
        axes[1, 0].text(0.5, 0.5, 'No cumulative IC yet', ha='center', va='center')
    else:
        axes[1, 0].plot(
            pd.to_datetime(primary_ic['trade_date']),
            primary_ic['rank_ic'].fillna(0.0).cumsum(),
            color='#e45756',
            linewidth=1.2,
        )
        axes[1, 0].axhline(0.0, color='black', linewidth=0.8, linestyle='--')
        axes[1, 0].set_title('Cumulative Rank IC (5d)')

    if quantile_summary.empty:
        axes[1, 1].axis('off')
        axes[1, 1].text(0.5, 0.5, 'No quantile return yet', ha='center', va='center')
    else:
        axes[1, 1].bar(quantile_summary['quantile'].astype(str), quantile_summary['avg_forward_return'], color='#72b7b2')
        axes[1, 1].axhline(0.0, color='black', linewidth=0.8, linestyle='--')
        axes[1, 1].set_title('Average 5-quantile return (5d)')

    if long_short.empty:
        axes[1, 2].axis('off')
        axes[1, 2].text(0.5, 0.5, 'No long-short series yet', ha='center', va='center')
    else:
        axes[1, 2].plot(pd.to_datetime(long_short['trade_date']), long_short['cum_return'], color='#b279a2', linewidth=1.2)
        axes[1, 2].axhline(0.0, color='black', linewidth=0.8, linestyle='--')
        axes[1, 2].set_title('Long-short cumulative return (Q5 - Q1)')

    for ax in axes.flat:
        ax.tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()


In [ ]:
pit_warning_table = build_pit_warning_table()
daily_panel = get_daily_panel(DATA_MODE)
factor_data = build_factor_data(daily_panel)
ic_frame = compute_rank_ic(factor_data)
ic_summary = summarize_ic(ic_frame)
quantile_summary, quantile_daily, quantile_pivot, long_short = compute_quantile_returns(factor_data)
stability_table = build_stability_table(ic_frame)

coverage_summary = pd.DataFrame(
    {
        'metric': [
            'rows',
            'trade_dates',
            'stocks',
            'universe_rows',
            'universe_ratio',
            'close_missing_rows',
            'suspend_rows',
        ],
        'value': [
            len(factor_data),
            factor_data['trade_date'].nunique(),
            factor_data['ts_code'].nunique(),
            int(factor_data['universe_mask'].sum()),
            factor_data['universe_mask'].mean(),
            int(factor_data['close'].isna().sum()),
            int(factor_data['is_suspend'].sum()),
        ],
    }
)

version_compare = ic_summary[ic_summary['forward_horizon'] == PRIMARY_FORWARD].copy()
version_compare['factor_version'] = version_compare['factor_version'].map(VERSION_LABELS)

print('PIT warning table')
print(pit_warning_table.to_string(index=False))
print('\nCoverage summary')
print(coverage_summary.to_string(index=False))
print('\nFactor data sample')
print(factor_data.head(10).to_string(index=False))
print('\nIC summary by version on 5d forward return')
print(version_compare[['factor_version', 'count', 'mean', 'win_rate', 'ic_ir']].to_string(index=False))
print('\nStability split for size-neutral 5d IC')
print(stability_table.to_string(index=False))
print('\nAverage quantile return (size-neutral, 5d)')
print(quantile_summary.to_string(index=False))


In [ ]:
plot_factor_diagnostics(factor_data, ic_frame, quantile_summary, long_short)

primary_slice = ic_summary[
    (ic_summary['factor_version'] == PRIMARY_FACTOR)
    & (ic_summary['forward_horizon'] == PRIMARY_FORWARD)
]

print('Key takeaways')
print(f"- universe rows: {int(factor_data['universe_mask'].sum())}")
if primary_slice.empty or int(factor_data['universe_mask'].sum()) == 0:
    print('- not enough valid rows yet for IC / quantile diagnostics')
    print('- if DATA_MODE=duckdb, current warehouse likely does not yet have enough 20d history and forward window')
else:
    primary_row = primary_slice.iloc[0]
    long_short_mean = long_short['long_short_return'].mean()
    long_short_hit = (long_short['long_short_return'] > 0).mean()
    print(f"- size-neutral 5d mean Rank IC: {primary_row['mean']:.4f}")
    print(f"- size-neutral 5d IC win rate: {primary_row['win_rate']:.2%}")
    print(f"- Q5-Q1 average 5d return: {long_short_mean:.4%}")
    print(f"- Q5-Q1 positive ratio: {long_short_hit:.2%}")


## 下一步

1. 把 `DATA_MODE` 从 `mock` 切到 `duckdb`，检查真实 `daily_panel` 是否能直接跑通。
2. 真实数据接入后，先逐项复核价格、复权、市值和停牌字段的 PIT 风险。
3. 如果这轮 notebook 口径稳定，再把稳定函数抽到 `src/`，形成可复用的因子研究模块。
